In [ ]:
from valdpy import ValdAuth, SmartSpeedAPI
from valdpy.utils import read_credentials

%load_ext autoreload
%autoreload 2

# SmartSpeed API Example

This example demonstrates how to use the VALDPY package to access SmartSpeed (timing gate system) test data.

In [ ]:
creds = read_credentials('vald_api_cred.txt')
client_id = creds['client_id']
client_secret = creds['client_secret']
tenant_id = creds['tenant_id']
print(f"Client ID: {client_id[:10]}...")
print(f"Tenant ID: {tenant_id}")

Client ID: 
Client Secret: 
Tenant ID: 


## Step 1: Authentication

In [ ]:
auth = ValdAuth(client_id, client_secret, tenant_id=tenant_id, region='USA')

### Get OAuth Access Token

In [ ]:
token = auth.get_token()
print(f"Token obtained: {token[:20]}...")

### Optional: Get Tenant Information

In [ ]:
all_tenants = auth.get_all_tenants()
print(f"Found {len(all_tenants)} tenant(s)")

[{'id': '61476d2d-917a-4b2e-b2f6-e43346d2cb66',
  'name': 'University of Oregon'}]

In [ ]:
tenant_info = auth.get_tenant_info()
print(f"Tenant Name: {tenant_info.get('name')}")
print(f"Tenant ID: {tenant_info.get('id')}")

{'id': '61476d2d-917a-4b2e-b2f6-e43346d2cb66',
 'name': 'University of Oregon',
 'sport': 'MultiSportCollege',
 'league': 'NCAA',
 'logoUri': 'https://content.valdperformance.com/logos/61476d2d-917a-4b2e-b2f6-e43346d2cb66.jpg'}

## Step 2: Get Categories and Groups

In [ ]:
categories_df = auth.get_tenant_categories()
print(categories_df[['name', 'id']].to_string(index=False))

,id,syncId,name
0,306f33af-2939-480b-bbd7-2ce643c888f0,None,Uncategorised
1,3a32a047-1292-49f1-a8aa-8f830ea58ac8,None,Position
2,1fdaf750-970a-4047-badc-945e6e6994cf,None,Team
3,2828d5d2-ce26-4297-b4c1-a16039889a86,None,Sport


In [ ]:
groups_df = auth.get_tenant_groups()
print(f"Found {len(groups_df)} group(s)")
print(groups_df[['name', 'id']].head(10).to_string(index=False))

## Step 3: Get Profiles

In [ ]:
group_name = 'Research'
category_name = 'Team'

try:
    profiles_df = auth.get_group_profiles(group_name=group_name, category_name=category_name)
    print(f"Found {len(profiles_df)} profile(s)")
    print(profiles_df[['firstName', 'lastName', 'profileId']].head(10).to_string(index=False))
except Exception as e:
    print(f"Error: {e}")

## Step 4: Initialize SmartSpeed API

In [ ]:
ss = SmartSpeedAPI(tenant_id=auth.tenant_id, header=auth.header, region='USA')

### Get Test Information

In [ ]:
date = '01/01/2025'
tests_df = ss.get_tests_info(date)

if tests_df is not None:
    print(f"Found {len(tests_df)} test(s)")
    print(tests_df[['testId', 'profileId']].head())
else:
    print("No tests found")

,id,testResultId,groupUnderTestId,profileId,testDateUtc,deviceCount,repCount,testTypeName,testName,isValid,...,reactiveDelayMinimumInSeconds,reactiveDelayMaximumInSeconds,events,durationInSeconds,lapCount,intervalType,testStandardType,dropHeight,dropHeightEnabled,weightKg
0,ddc07ed7-750d-40bf-89cd-50a5447437b0,8d5645dd-4190-4215-8482-25ece1c449df,None,6dac6c2d-b628-4681-a1cf-0987cbe70ea7,2025-01-15T22:54:25,4,1,OneWay,30 m WSOC,True,...,0.0,0.0,None,None,1,None,Standard,0.0,False,None
1,40da1ed2-a204-44a5-a33b-93807ddbe38a,a9de110a-8436-4fbd-8106-46675c67d07b,None,6dac6c2d-b628-4681-a1cf-0987cbe70ea7,2025-01-15T22:42:47,4,1,OneWay,30 m WSOC,True,...,0.0,0.0,None,None,1,None,Standard,0.0,False,None


In [ ]:
if tests_df is not None and len(tests_df) > 0:
    test_id = tests_df.iloc[0]['testId']
    results_df = ss.get_test_results(test_id)
    if results_df is not None:
        print(results_df.head())
    else:
        print("No results available")